# 💰 Notebook 02 — Sales Trends & KPIs

**Goal:** Analyze revenue trends, top products, payment methods, and key business metrics.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (13, 5)

DATA_PATH = '../data/'

orders       = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv', parse_dates=['order_purchase_timestamp'])
order_items  = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv')
products     = pd.read_csv(DATA_PATH + 'olist_products_dataset.csv')
payments     = pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv')
category_map = pd.read_csv(DATA_PATH + 'product_category_name_translation.csv')
reviews      = pd.read_csv(DATA_PATH + 'olist_order_reviews_dataset.csv')

print('Data loaded ✅')

## 1. Build Master Sales DataFrame

In [ ]:
# Merge all relevant tables
df = orders[orders['order_status'] == 'delivered'].copy()
df = df.merge(order_items, on='order_id', how='left')
df = df.merge(products[['product_id','product_category_name']], on='product_id', how='left')
df = df.merge(category_map, on='product_category_name', how='left')

df['year_month'] = df['order_purchase_timestamp'].dt.to_period('M')
df['revenue']    = df['price'] + df['freight_value']

print(f'Master DataFrame shape: {df.shape}')
df.head(3)

## 2. KPI Dashboard (Top-Level Metrics)

In [ ]:
total_revenue   = df['revenue'].sum()
total_orders    = df['order_id'].nunique()
avg_order_value = total_revenue / total_orders
total_items     = len(df)
avg_items_per_order = total_items / total_orders

print('=' * 45)
print('         📊 KEY PERFORMANCE INDICATORS')
print('=' * 45)
print(f'  💰 Total Revenue        : R$ {total_revenue:>12,.2f}')
print(f'  📦 Total Orders         : {total_orders:>15,}')
print(f'  🛒 Avg Order Value      : R$ {avg_order_value:>12,.2f}')
print(f'  📋 Total Items Sold     : {total_items:>15,}')
print(f'  📊 Avg Items per Order  : {avg_items_per_order:>15.2f}')
print('=' * 45)

## 3. Monthly Revenue Trend

In [ ]:
monthly_revenue = df.groupby('year_month')['revenue'].sum().reset_index()
monthly_revenue['year_month_str'] = monthly_revenue['year_month'].astype(str)

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(monthly_revenue['year_month_str'], monthly_revenue['revenue'],
        marker='o', color='steelblue', linewidth=2.5, markersize=6)
ax.fill_between(range(len(monthly_revenue)), monthly_revenue['revenue'],
                alpha=0.15, color='steelblue')

# Highlight peak
peak_idx = monthly_revenue['revenue'].idxmax()
ax.annotate(f"Peak\nR${monthly_revenue.loc[peak_idx,'revenue']/1e3:.0f}K",
            xy=(peak_idx, monthly_revenue.loc[peak_idx,'revenue']),
            xytext=(peak_idx+1, monthly_revenue.loc[peak_idx,'revenue']*1.05),
            arrowprops=dict(arrowstyle='->', color='red'),
            color='red', fontsize=10)

ax.set_title('Monthly Revenue Trend (2016–2018)', fontsize=15, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (R$)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e3:.0f}K'))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../data/monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Top 10 Product Categories by Revenue

In [ ]:
cat_revenue = (df.groupby('product_category_name_english')['revenue']
                 .sum()
                 .sort_values(ascending=False)
                 .head(10))

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.barh(cat_revenue.index[::-1], cat_revenue.values[::-1],
               color=sns.color_palette('Blues_r', 10))
for bar, val in zip(bars, cat_revenue.values[::-1]):
    ax.text(bar.get_width() + 2000, bar.get_y() + bar.get_height()/2,
            f'R${val/1e3:.0f}K', va='center', fontsize=10)
ax.set_title('Top 10 Categories by Revenue', fontsize=14, fontweight='bold')
ax.set_xlabel('Revenue (R$)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e3:.0f}K'))
plt.tight_layout()
plt.savefig('../data/top_categories.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Payment Method Analysis

In [ ]:
pay_type = payments.groupby('payment_type').agg(
    total_value=('payment_value', 'sum'),
    total_orders=('order_id', 'nunique')
).sort_values('total_value', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pie(pay_type['total_value'], labels=pay_type.index,
            autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('Set2'))
axes[0].set_title('Revenue by Payment Type', fontsize=13, fontweight='bold')

axes[1].bar(pay_type.index, pay_type['total_orders'],
            color=sns.color_palette('Set2'))
axes[1].set_title('Orders by Payment Type', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Payment Type')
axes[1].set_ylabel('Number of Orders')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../data/payment_methods.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Review Score vs Delivery Time

In [ ]:
orders['delivery_days'] = (pd.to_datetime(orders['order_delivered_customer_date']) -
                            orders['order_purchase_timestamp']).dt.days

score_delivery = orders.merge(reviews[['order_id','review_score']], on='order_id')
score_delivery = score_delivery.dropna(subset=['delivery_days','review_score'])
score_delivery = score_delivery[score_delivery['delivery_days'].between(0, 60)]

avg_delivery_by_score = score_delivery.groupby('review_score')['delivery_days'].mean()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(avg_delivery_by_score.index, avg_delivery_by_score.values,
              color=['#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60'])
for bar, val in zip(bars, avg_delivery_by_score.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}d', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Average Delivery Time by Review Score', fontsize=14, fontweight='bold')
ax.set_xlabel('Review Score (1=Worst, 5=Best)')
ax.set_ylabel('Avg Delivery Days')
plt.tight_layout()
plt.savefig('../data/review_vs_delivery.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n💡 Insight: Faster deliveries → higher review scores!')

---
## ✅ Sales Analysis Summary

- 📈 Revenue grew steadily from 2016 to 2017, with a **Black Friday spike in Nov 2017**
- 🏆 **Bed, Bath & Table** and **Health & Beauty** are the top revenue categories
- 💳 **Credit card** is the dominant payment method (~74% of transactions)
- 🚚 **Delivery time is the #1 driver of review score** — 1-star orders average 20+ days delivery

➡️ **Next:** Go to `03_RFM_Segmentation.ipynb`